# ZiaFlow connection and data-quality audit

## Summary
October 19–25, 2026, Phoenix; current day partial. Collection begins October 22. Intake flow is unproven for the deployed Web3Forms path. GA4 reporting and seven-day GSC totals are unknown. See the adjacent dated Markdown plan for findings and sources.

## Context and methods
Supabase aggregates and GSC period aggregates are controlling sources. Do not sum overlapping snapshots, join GSC to people, or treat tests as business. No live provider requests are made by this notebook.

## Data: exact read-only connector queries

```sql
select (started_at at time zone 'America/Phoenix')::date as date,
 count(*) filter(where status='success') as successful_runs,
 count(*) filter(where status='failed') as failed_runs
from public.clarity_sync_runs
where started_at >= '2026-10-19 00:00:00 America/Phoenix'
 and started_at < '2026-10-26 00:00:00 America/Phoenix'
group by 1 order by 1;

select date, sum(confirmed_form_submissions) as confirmed_forms,
 sum(test_observations) as excluded_tests
from public.website_confirmed_leads_daily
where date between '2026-10-19' and '2026-10-25'
group by date order by date;

select source, property, period_start, period_end, grain, captured_at,
 payload->'totals' as totals, payload->'access' as access, limitations
from public.analytics_source_snapshots
where source='gsc' and property='sc-domain:ziaflow.com'
order by captured_at desc limit 8;
```

In [1]:
syncs = [("2026-10-22",4,0),("2026-10-23",9,1),("2026-10-24",9,1),("2026-10-25 partial",8,0)]
assert sum(r[1] for r in syncs) == 30
assert sum(r[2] for r in syncs) == 2
for date, success, failure in syncs:
    print(date, "successful syncs:", success, "failed syncs:", failure)
print("Overall successful syncs:", sum(r[1] for r in syncs))
print("Confirmed non-test inquiries: 2; excluded tests: 5")
print("Seven-day GSC traffic and GA4 sessions: unknown")

2026-10-22 successful syncs: 4 failed syncs: 0
2026-10-23 successful syncs: 9 failed syncs: 1
2026-10-24 successful syncs: 9 failed syncs: 1
2026-10-25 partial successful syncs: 8 failed syncs: 0
Overall successful syncs: 30
Confirmed non-test inquiries: 2; excluded tests: 5
Seven-day GSC traffic and GA4 sessions: unknown


## Takeaways
Fix and verify intake and event contracts before acquisition expansion. 30 successful syncs are pipeline health, not 30 visits. Missing October 19–21 observations remain unknown. The 28-day GSC snapshot has 4 clicks and 422 impressions; it is not the requested seven-day report. No plot is needed for this small diagnostic audit table. The notebook replays reviewed inputs; live queries require the authorized connector.